# Satellite CNN vs Transformer Across Source Scenes

**Research question:** How do a small CNN and a spatial CNN-Transformer generalize to an unseen source scene?

Kerem Ozcan | IBM AI Engineering portfolio study | Prepared with AI assistance

This independently authored experiment applies course concepts. It is not an IBM assessment notebook. Scores below are produced by the current run; no supplied course predictions/checkpoints are used.

## Experiment design

Reserve T32UPE for testing and T31UDQ for validation. Train both fresh models for three epochs with shared stem initialization and identical image order/augmentation. Restore the validation-selected checkpoints. The hybrid retains 16 spatial tokens plus CLS; parameter counts differ, so this is not an attention-only ablation.

**Evaluation boundary:** Only four source-scene groups and one seed; the validation group has 93 images. Exact decoded-image duplicates are checked, but geographic overlap and near-duplicates are unverified. Three epochs do not establish architecture superiority.

## Runtime

Kaggle internet access is required for provider-hosted data and public model downloads. Retain Kaggle's preinstalled Torch/TensorFlow builds; the cell pins the Transformers/PEFT interfaces used here. CPU is supported; an enabled GPU accelerates the Torch projects. Actual package versions and source hashes are saved with the metrics.

In [ ]:
import os
import subprocess
import sys
os.environ["USE_TF"] = "0"  # Transformers uses PyTorch; the aircraft module imports TF separately.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.45.2", "peft==0.11.1", "accelerate>=0.30,<2"], check=True)
from pathlib import Path
import json
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()/".lab-cache/kaggle-local"
WORK.mkdir(parents=True, exist_ok=True)
SCRATCH = Path("/kaggle/temp/ibm-portfolio") if Path("/kaggle/working").is_dir() else WORK/"scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)


## Data provenance

Data: [IBM Skills Network archive](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/4Z1fwRR295-1O3PMQBH6Dg/images-dataSAT.tar). The original image filenames identify Sentinel-2 scenes. Dataset redistribution permission and broader geographic overlap are not established; data are fetched from the provider, not republished here.

Expected download SHA-256: `5e45d64a33052c2a7f2e8b8d4232fbba6b7419831d192de56cae87dbe9547206`. The downloader rejects changed bytes and unsafe tar members.

In [ ]:
"""Small, standalone data acquisition helpers embedded in the Kaggle releases."""

import hashlib
import shutil
import tarfile
import urllib.request
from pathlib import Path


def verified_download(url: str, target: Path, expected_sha256: str) -> Path:
    """Download to a temporary file and accept only the reviewed dataset bytes."""
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and hashlib.sha256(target.read_bytes()).hexdigest() == expected_sha256:
        return target
    temporary = target.with_suffix(target.suffix + ".download")
    try:
        with urllib.request.urlopen(url, timeout=60) as response, temporary.open("wb") as stream:
            shutil.copyfileobj(response, stream)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != expected_sha256:
            raise ValueError("Downloaded data changed; review its provenance before running.")
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)
    return target


def unpack_regular_tar(archive: Path, destination: Path) -> None:
    """Reject links, special files and traversal before writing any archive member."""
    root = destination.resolve()
    with tarfile.open(archive) as bundle:
        members = bundle.getmembers()
        for member in members:
            path = root / member.name
            if (
                Path(member.name).is_absolute()
                or not path.resolve().is_relative_to(root)
                or not (member.isfile() or member.isdir())
            ):
                raise ValueError("Unsafe dataset archive member.")
        for member in members:
            path = root / member.name
            if member.isdir():
                path.mkdir(parents=True, exist_ok=True)
            else:
                path.parent.mkdir(parents=True, exist_ok=True)
                with bundle.extractfile(member) as source, path.open("wb") as target:
                    shutil.copyfileobj(source, target)


In [ ]:
# Fetch from the provider; reject changed archives by SHA-256.
DATA_DIR = SCRATCH / "inputs" / 'satellite'
DATA_PATH = DATA_DIR / 'images_dataSAT'
USING_LOCAL_CACHE = False
if not Path("/kaggle/working").is_dir():
    local_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/"provenance/source-manifest.json").exists()), None)
    if local_root is not None:
        candidate = local_root / '.lab-cache/06-computer-vision-capstone/lab-vision-transformers-using-pytorch/images_dataSAT'
        if candidate.exists():
            DATA_PATH = candidate
            USING_LOCAL_CACHE = True
if not USING_LOCAL_CACHE:
    archive = verified_download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/4Z1fwRR295-1O3PMQBH6Dg/images-dataSAT.tar', DATA_DIR/'images-dataSAT.tar', '5e45d64a33052c2a7f2e8b8d4232fbba6b7419831d192de56cae87dbe9547206')
    if archive.suffix == ".tar":
        unpack_regular_tar(archive, DATA_DIR)
    else:
        DATA_PATH = archive
if not DATA_PATH.exists():
    raise FileNotFoundError("Expected dataset layout was not produced.")
OUTPUT = WORK / 'satellite'
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT = SCRATCH / "runs" / 'satellite'


## Implementation

Expand this cell to inspect the complete standalone implementation. The split manifests, actual processed sample counts and selected-checkpoint reload checks are part of the experiment.

In [ ]:
"""Shared provenance and evaluation helpers for the selected portfolio projects."""

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import random
from collections import Counter, defaultdict
from datetime import UTC, datetime
from pathlib import Path


def file_sha256(path: Path) -> str:
    """Hash an input/checkpoint without reading the entire file into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path: Path, value: dict | list) -> None:
    """Publish a complete JSON artifact atomically."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8"
    )
    temporary.replace(path)


def runtime(packages: list[str]) -> dict:
    """Capture versions without collecting environment variables or host paths."""
    return {
        "utc": datetime.now(UTC).isoformat(),
        "python": platform.python_version(),
        "platform": platform.system(),
        "packages": {name: importlib.metadata.version(name) for name in packages},
    }


def source_identity(project_file: str | None, embedded_digest: str | None = None) -> dict:
    """Bind module runs and generated notebooks to their authored source files."""
    project_digest = file_sha256(Path(project_file)) if project_file else embedded_digest
    common_digest = (
        file_sha256(Path(__file__))
        if "__file__" in globals()
        else globals().get("COMMON_SOURCE_SHA256")
    )
    if not project_digest or not common_digest:
        raise ValueError("Project source identity is missing.")
    return {"project_sha256": project_digest, "common_sha256": common_digest}


def index_images(
    root: Path, class_names: list[str], directories: dict[str, str] | None = None
) -> list[dict]:
    """Index decoded RGB content as well as bytes, retaining only relative filenames."""
    from PIL import Image, ImageOps

    rows = []
    directories = directories or {"unsplit": "."}
    for role, directory in directories.items():
        for label, name in enumerate(class_names):
            folder = root / directory / name
            if not folder.is_dir():
                raise ValueError(f"Missing image class directory: {directory}/{name}")
            for path in sorted(folder.rglob("*")):
                if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                    continue
                with Image.open(path) as original:
                    image = ImageOps.exif_transpose(original).convert("RGB")
                    pixels = str(image.size).encode() + image.tobytes()
                rows.append(
                    {
                        "file": path.relative_to(root).as_posix(),
                        "label": label,
                        "role": role,
                        "file_sha256": file_sha256(path),
                        "pixel_sha256": hashlib.sha256(pixels).hexdigest(),
                    }
                )
    if not rows:
        raise ValueError("No images found.")
    return rows


def clean_image_splits(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    """Remove exact duplicates and conflicting labels, preserving held-out copies first."""
    groups = defaultdict(list)
    for row in rows:
        groups[row["pixel_sha256"]].append(row)
    retained, removed = [], []
    priority = {"test": 0, "validation": 1, "train": 2}
    for group in groups.values():
        if len({row["label"] for row in group}) > 1:
            removed.extend(
                {"file": row["file"], "reason": "conflicting_pixel_labels"} for row in group
            )
            continue
        ordered = sorted(group, key=lambda row: (priority[row["role"]], row["file"]))
        retained.append(ordered[0])
        removed.extend(
            {"file": row["file"], "reason": "exact_pixel_duplicate"} for row in ordered[1:]
        )
    return sorted(retained, key=lambda row: row["file"]), removed


def balanced_subset(rows: list[dict], per_class: int | None, seed: int) -> list[dict]:
    """Select a reproducible pilot without changing the complete split manifest."""
    if per_class is None:
        return rows
    rng = random.Random(seed)
    result = []
    for label in sorted({row["label"] for row in rows}):
        group = sorted((row for row in rows if row["label"] == label), key=lambda row: row["file"])
        rng.shuffle(group)
        result.extend(group[:per_class])
    return sorted(result, key=lambda row: row["file"])


def split_counts(rows: list[dict]) -> dict:
    """Describe the actual records, including class support."""
    return {
        role: dict(Counter(str(row["label"]) for row in rows if row["role"] == role))
        for role in ["train", "validation", "test"]
    }


def classification_metrics(labels: list[int], probabilities: list[float]) -> dict:
    """Evaluate exactly the predictions supplied, never the size of an upstream dataset."""
    import numpy as np
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        roc_auc_score,
    )

    if not labels or len(labels) != len(probabilities):
        raise ValueError("Predictions and labels must have the same nonzero length.")
    probabilities = np.asarray(probabilities)
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Expected finite probabilities in [0, 1].")
    predicted = (probabilities >= 0.5).astype(int)
    return {
        "samples": len(labels),
        "accuracy": float(accuracy_score(labels, predicted)),
        "macro_f1": float(
            f1_score(labels, predicted, labels=[0, 1], average="macro", zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(labels, probabilities)) if len(set(labels)) == 2 else None,
        "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist(),
        "per_class": classification_report(
            labels, predicted, labels=[0, 1], output_dict=True, zero_division=0
        ),
    }

COMMON_SOURCE_SHA256 = 'ded7059d91e2123ef7bc9be2823a1d220d905041f45b9dbe83bac59d27547853'
PROJECT_SOURCE_SHA256 = 'bf1d0613411d582ae60786fc63080e3c90be901a8879ecd33f6555250e53bdb4'
"""Train a fresh CNN and a genuinely spatial CNN-Transformer on scene-separated images."""


import argparse
import copy
import hashlib
import random
import re
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset



def scene_id(filename: str) -> str:
    """Use the complete source scene, excluding the final tile number."""
    return re.sub(r"_\d+\.[^.]+$", "", Path(filename).name)


def assign_scene_splits(
    rows: list[dict], test_scene: str = "T32UPE", validation_scene: str = "T31UDQ"
) -> list[dict]:
    """Reserve whole source-scene groups; never fall back silently to a random split."""
    if test_scene == validation_scene:
        raise ValueError("Validation and test scenes must differ.")
    result = []
    for row in rows:
        scene = scene_id(row["file"])
        role = (
            "test"
            if test_scene in scene
            else "validation"
            if validation_scene in scene
            else "train"
        )
        result.append({**row, "scene": scene, "role": role})
    for role in ["train", "validation", "test"]:
        if {row["label"] for row in result if row["role"] == role} != {0, 1}:
            raise ValueError(f"{role} must contain both classes; inspect the chosen scenes.")
    return result


class SatelliteDataset(Dataset):
    """Deterministic augmentation per image/epoch, independent of model dropout RNG."""

    def __init__(self, root: Path, rows: list[dict], training: bool, seed: int):
        self.root, self.rows, self.training, self.seed = root, rows, training, seed
        self.epoch = 0

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(self.root / row["file"]) as source:
            image = ImageOps.exif_transpose(source).convert("RGB").resize((64, 64))
        token = f"{self.seed}:{self.epoch}:{row['file']}".encode()
        if self.training and hashlib.sha256(token).digest()[0] < 128:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        values = np.asarray(image, dtype=np.float32) / 255.0
        return torch.from_numpy(values.transpose(2, 0, 1).copy()), row["label"], row["file"]


class Stem(nn.Sequential):
    """Preserve an 8x8 feature map for a 64x64 image."""

    def __init__(self):
        layers = []
        for before, after in [(3, 16), (16, 32), (32, 64)]:
            layers.extend(
                [
                    nn.Conv2d(before, after, 3, stride=2, padding=1),
                    nn.GroupNorm(4, after),
                    nn.GELU(),
                ]
            )
        super().__init__(*layers)


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = Stem()
        self.head = nn.Linear(64, 2)

    def forward(self, images):
        return self.head(self.stem(images).mean(dim=(-1, -2)))


class SpatialHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = Stem()
        self.patch = nn.Conv2d(64, 64, 2, stride=2)
        self.cls = nn.Parameter(torch.zeros(1, 1, 64))
        self.position = nn.Parameter(torch.zeros(1, 17, 64))
        layer = nn.TransformerEncoderLayer(
            64, 4, dim_feedforward=128, dropout=0.1, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, 2)
        self.head = nn.Linear(64, 2)

    def tokens(self, images):
        patches = self.patch(self.stem(images)).flatten(2).transpose(1, 2)
        if patches.shape[1] <= 1:
            raise ValueError("Spatial attention requires multiple image patches.")
        return torch.cat([self.cls.expand(len(images), -1, -1), patches], dim=1) + self.position

    def forward(self, images):
        return self.head(self.encoder(self.tokens(images))[:, 0])


def evaluate(model, loader, device) -> dict:
    """Compute weighted loss/metrics over actual evaluated examples."""
    model.eval()
    labels, probabilities, ids = [], [], []
    loss_sum = 0.0
    with torch.no_grad():
        for images, target, filenames in loader:
            logits = model(images.to(device))
            loss_sum += nn.functional.cross_entropy(
                logits, target.to(device), reduction="sum"
            ).item()
            labels.extend(target.tolist())
            probabilities.extend(logits.softmax(-1)[:, 1].cpu().tolist())
            ids.extend(filenames)
    result = classification_metrics(labels, probabilities)
    return {**result, "loss": loss_sum / len(labels), "ids": ids}


def restore(model, path: Path, device):
    """Restore all weights strictly before final evaluation."""
    model.load_state_dict(torch.load(path, map_location=device, weights_only=True), strict=True)
    return model.eval()


def run(
    data_root: Path, output: Path, *, full: bool = False, seed: int = 42, epochs: int = 3
) -> dict:
    """Run a bounded pilot by default; a full run keeps every retained image."""
    torch.set_num_threads(1)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rows = assign_scene_splits(index_images(data_root, ["class_0_non_agri", "class_1_agri"]))
    rows, removed = clean_image_splits(rows)
    write_json(output / "split-manifest.json", {"records": rows, "removed": removed})
    groups = {
        role: [row for row in rows if row["role"] == role]
        for role in ["train", "validation", "test"]
    }
    for role in groups:
        if {r["label"] for r in groups[role]} != {0, 1}:
            raise ValueError(f"Both classes must remain in {role} after cleaning.")
    selected = {
        role: balanced_subset(group, None if full else (16 if role == "train" else 8), seed)
        for role, group in groups.items()
    }
    validation = DataLoader(
        SatelliteDataset(data_root, selected["validation"], False, seed), batch_size=8
    )
    test = DataLoader(SatelliteDataset(data_root, selected["test"], False, seed), batch_size=8)
    torch.manual_seed(seed)
    initial_stem = copy.deepcopy(Stem().state_dict())
    results = {}
    for name, factory in [("cnn", CNN), ("spatial_hybrid", SpatialHybrid)]:
        torch.manual_seed(seed)
        model = factory().to(device)
        model.stem.load_state_dict(initial_stem)
        dataset = SatelliteDataset(data_root, selected["train"], True, seed)
        train = DataLoader(
            dataset, batch_size=8, shuffle=True, generator=torch.Generator().manual_seed(seed)
        )
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
        history, visited = [], []
        best_f1 = -1.0
        checkpoint = output / name / "best.pt"
        checkpoint.parent.mkdir(parents=True, exist_ok=True)
        started = time.perf_counter()
        for epoch in range(epochs if full else 1):
            dataset.epoch = epoch
            model.train()
            total_loss, processed = 0.0, 0
            for images, labels, filenames in train:
                optimizer.zero_grad()
                loss = nn.functional.cross_entropy(model(images.to(device)), labels.to(device))
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(labels)
                processed += len(labels)
                visited.extend(filenames)
            score = evaluate(model, validation, device)
            history.append(
                {
                    "epoch": epoch + 1,
                    "train_loss": total_loss / processed,
                    "validation_macro_f1": score["macro_f1"],
                    "train_samples": processed,
                }
            )
            if score["macro_f1"] > best_f1:
                best_f1 = score["macro_f1"]
                torch.save(model.state_dict(), checkpoint)
        restore(model, checkpoint, device)
        probe = next(iter(validation))[0].to(device)
        with torch.no_grad():
            expected = model(probe).cpu()
            actual = restore(factory().to(device), checkpoint, device)(probe).cpu()
        torch.testing.assert_close(expected, actual, rtol=0, atol=0)
        results[name] = {
            "parameters": sum(p.numel() for p in model.parameters()),
            "spatial_tokens": 16 if name == "spatial_hybrid" else None,
            "seconds": time.perf_counter() - started,
            "history": history,
            "checkpoint_sha256": file_sha256(checkpoint),
            "exact_reload": True,
            "training_ids": visited,
            "test": evaluate(model, test, device),
        }
    if results["cnn"]["training_ids"] != results["spatial_hybrid"]["training_ids"]:
        raise AssertionError("Compared models saw different sample orders.")
    report = {
        "project": "satellite",
        "profile": "full" if full else "bounded-pilot",
        "seed": seed,
        "device": str(device),
        "complete_split_counts": split_counts(rows),
        "selected_split_counts": split_counts([r for group in selected.values() for r in group]),
        "split_manifest_sha256": file_sha256(output / "split-manifest.json"),
        "comparison": "Different parameter counts; shared stem initialization and sample order. Not an isolated attention-only ablation.",
        "near_duplicate_or_geographic_overlap_audited": False,
        "models": results,
        "source": source_identity(
            globals().get("__file__"), globals().get("PROJECT_SOURCE_SHA256")
        ),
        "runtime": runtime(["torch", "numpy", "scikit-learn", "Pillow"]),
    }
    write_json(output / "metrics.json", report)
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-root", type=Path, required=True)
    parser.add_argument(
        "--output", type=Path, default=Path(".lab-cache/selected-projects/satellite")
    )
    parser.add_argument("--full", action="store_true")
    args = parser.parse_args()
    report = run(args.data_root, args.output, full=args.full)
    print({name: row["test"]["macro_f1"] for name, row in report["models"].items()})



In [ ]:
report = run(DATA_PATH, RUN_OUTPUT, full=True, seed=42)
for filename in ['metrics.json', 'split-manifest.json']:
    shutil.copyfile(RUN_OUTPUT/filename, OUTPUT/filename)
print('Executed profile:', report['profile'])
print('Evaluated split sizes:', report['selected_split_counts'])
print('Runtime:', report['runtime'])

## Measured results

Read the sample counts and stated limitations alongside these measurements.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, result) in zip(axes, report["models"].items()):
    matrix = result["test"]["confusion_matrix"]
    ax.imshow(matrix, cmap="Blues")
    for i in range(2):
        for j in range(2): ax.text(j, i, str(matrix[i][j]), ha="center", va="center")
    ax.set(title=f"{name}: macro F1 {result['test']['macro_f1']:.3f}", xlabel="Predicted", ylabel="True", xticks=[0,1], yticks=[0,1])
    print(name, "parameters", result["parameters"], "test samples", result["test"]["samples"], "reload", result["exact_reload"])
fig.tight_layout(); fig.savefig(OUTPUT/"results.png", dpi=160); plt.show()
print("Labels: 0 = non-agricultural, 1 = agricultural")


## What this experiment establishes

The saved metrics bind this execution to its source hashes, dataset partitions and runtime versions. Checkpoint reload is an implementation check. It does not establish task quality by itself.

Only four source-scene groups and one seed; the validation group has 93 images. Exact decoded-image duplicates are checked, but geographic overlap and near-duplicates are unverified. Three epochs do not establish architecture superiority.

## Attribution

Data: [IBM Skills Network archive](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/4Z1fwRR295-1O3PMQBH6Dg/images-dataSAT.tar). The original image filenames identify Sentinel-2 scenes. Dataset redistribution permission and broader geographic overlap are not established; data are fetched from the provider, not republished here.

Concepts studied through the [IBM AI Engineering Professional Certificate](https://www.coursera.org/professional-certificates/ai-engineer). Source/data/model licenses remain with their providers. No general license is applied to IBM course materials or third-party datasets.